# Bigram Language Model

The picture

Imagine a shelf with one weighted die per character. Each die has a face for every character in the alphabet, weighted by how often that face followed this character in your text. To generate text, you look at the last character, pick up its die, roll it, write down the result, and repeat.

That is the whole model. Training means adjusting the weights on the dice so they match the text.

Where it breaks: a real bigram model has no counting step. It starts with random weights and learns them by gradient descent. The optimum happens to equal the counted frequencies, and that fact is your best verification tool (see below).

**The parts and their jobs**

- **Vocabulary (stoi, itos)**: Gives each character a number, i.e. which die is which
- **(x, y) batches**: y is x shifted left by one: the character that actually came next
- **nn.Embedding(V, V)**: The shelf of dice. Row i holds logits (unnormalised scores) for what follows character i
- **Softmax**: Turns a row of scores into probabilities, i.e. weights the die
- **Cross-entropy**: Measures how surprised the die is by what really came next
- **AdamW Nudges**: the table to reduce that surprise
- **Sampling loop**: Rolls the dice, one character at a time

## Init

In [3]:
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
INPUT_DIR = Path("../data")
tinyshakespears_path = INPUT_DIR / "tinyshakespears.txt"

## Load Data

In [8]:
with open(tinyshakespears_path, "r") as f:
    text = f.read()

chars = sorted(set(text))
print(len(chars))

65


In [20]:
VOCAB_SIZE = 65

In [10]:
# Mapping
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [32]:
# Put the whole text in one torch.long tensor, with the first 90% for training and the rest for validation.
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("Data:", data.shape)
print("Train data:", train_data.shape)
print("Val data:", val_data.shape)

Data: torch.Size([1115394])
Train data: torch.Size([1003854])
Val data: torch.Size([111540])


## Batching

**Shapes**. With batch size B and context length T (block_size):

- x: (B, T) integers
- y: (B, T) integers, the same sequence shifted by one
- logits = table(x): (B, T, V), one row of scores for every position

Pick B random starting offsets, and for each take a window of T = 8 characters. y is the same window shifted by one.

In [22]:
ix = torch.randint(len(data) - 8, (512,))
x = torch.stack([data[i:i+8] for i in ix])
y = torch.stack([data[i+1:i+8+1] for i in ix])

print("Index shape:", ix.shape)
print("Input shape:", x.shape)
print("Target shape:", y.shape)

Index shape: torch.Size([512])
Input shape: torch.Size([512, 8])
Target shape: torch.Size([512, 8])


## Model

In [21]:
model = nn.Embedding(VOCAB_SIZE, VOCAB_SIZE)
logits = model(x)
print("Logits shape:", logits.shape)

Logits shape: torch.Size([512, 8, 65])


In [24]:
# Flattening for cross-entropy loss
B, T, C = logits.shape

logits_flat = logits.view(B*T, C)
target_flat = y.view(B*T)

loss = F.cross_entropy(logits_flat, target_flat)
print("Cross-entropy Loss:", loss)

Cross-entropy Loss: tensor(4.6558, grad_fn=<NllLossBackward0>)


![Bigram Architecture](../assets/img/archi.jpg)

In [26]:
# Model class
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size: int):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, x, y=None):
        logits = self.token_embedding_table(x)
        if y is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            y_flat = y.view(B*T)
            loss = F.cross_entropy(logits_flat, y_flat)

        return logits, loss

## Training Loop

- Optimiser: AdamW
- Learning rate: 1e-2

In [27]:
model = BigramLanguageModel(VOCAB_SIZE)
optimiser = torch.optim.AdamW(model.parameters(), lr=1e-2)

In [29]:
# The loop
batch_size = 512
block_size = 8
max_iters = 1000

for steps in range(max_iters):
    # Random indices
    ix = torch.randint(len(train_data) - block_size, (batch_size,))
    x = torch.stack([train_data[i:i+block_size] for i in ix])
    y = torch.stack([train_data[i+1:i+block_size+1] for i in ix])

    # Forward
    logits, loss = model.forward(x, y)

    # Backward
    optimiser.zero_grad(set_to_none=True)
    loss.backward()
    optimiser.step()

    if steps % 100 == 0:
        print(f"Step {steps}: loss {loss.item():.4f}")

print("Training complete! Final loss:", loss.item())

Step 0: loss 4.6960
Step 100: loss 3.5217
Step 200: loss 2.9057
Step 300: loss 2.6585
Step 400: loss 2.5308
Step 500: loss 2.5158
Step 600: loss 2.4733
Step 700: loss 2.5093
Step 800: loss 2.4881
Step 900: loss 2.4564
Training complete! Final loss: 2.4691665172576904


## Sampling
Start with a context of shape (1, 1). Loop: get the logits, take only the last position, apply softmax, draw with `torch.multinomial`, and concatenate the result onto the context. Wrap it in `torch.no_grad()`.

Imagine an old-fashioned fortune-teller machine at a carnival. It prints a fortune one single letter at a time on a strip of paper.

- To print the second letter, the machine must feed the first letter back into its input slot.
- To print the third letter, it feeds both the first and second letters back in.

This looping, feed-forward process is called **autoregression** (regression on itself). In our bigram model, the sampling loop acts like this mechanical conveyor belt: it takes what it just wrote, looks at the last character, rolls a weighted die (`torch.multinomial`) based on the softmax probabilities, appends the new character to the running text context, and repeats the cycle.

![Sampling architecture](../assets/img/sampling.jpg)

In [30]:
@torch.no_grad()
def generate(model, idx, max_new_tokens):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):
        # Forward
        logits, loss = model(idx)

        # Get last time step: (B, C)
        logits = logits[:, -1, :]

        # Apply softmax to get proba: (B, C)
        probs = F.softmax(logits, dim=-1)

        # Sample from the distribution: (B, 1)
        idx_next = torch.multinomial(probs, num_samples=1)

        # Append sampled index to the running context sequence long
        idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    
    return idx

In [31]:
# Start with a single zero token (shape 1, 1)
context = torch.zeros((1, 1), dtype=torch.long)

# Generate 500 new tokens and decode them back to text
generated_indices = generate(model, context, max_new_tokens=500)[0].tolist()
print(decode(generated_indices))


Busourucorilay,


'd ppd f w
GRESt t'e ms the sises,
Thalo'siseUMy tond wot qurd!
Sirsco e the. unghe sofrk, d tho, imowan hallof hio w:
Yove ourstyos IUSthallen owhakither g f avindpowecinea isenQUCl shede un$CKE: ho geto maimoree'sirnturd oeromanor st thatNSt:

DUSQUSongnalyollkeof irar.
QURDilior ague tave tothe et y ooruemangn ot.
se.
LI h bellerint ot s, t Chal ANounomy wimitasthe
RIten t nd whepen kn palle

We ; oushithayop st mefig toupr-sthowighein heinfouton le hace!
Thtases onced y t h


## Building Count Matrix and Normalisation Rows

Imagine opening a massive accounting ledger where the rows are the current character and the columns are the next character. Every time you read a pair of characters in Shakespeare's text (like "th" or "he"), you walk to row 't' and add a tally mark under column 'h'.

- The Count Matrix ($N$): A $65 \times 65$ grid recording raw tally marks.
- Row Normalisation: Converting raw tally marks into exact percentages (probabilities) so that all possible next characters in a given row sum up to $1.0$.
- Validation Data: A separate, unseen segment of text held back like a final exam. If your count matrix learned the patterns well from the training text, it should assign high probabilities to character pairs in the validation set.

In [33]:
# Init count matrix with 1s for smoothing (65, 65)
N = torch.ones((VOCAB_SIZE, VOCAB_SIZE), dtype=torch.float32)

# Count bigram frequencies in the training data
for t1, t2 in zip(train_data[:-1], train_data[1:]):
    N[t1, t2] += 1

# Normalise rows to sum up to 1.0
P = N / N.sum(dim=1, keepdim=True)

## Evaluate Average Negative Log-Proba on Val Data

The negative log-likelihood (NLL) measures how surprised the model is by the true text. Lower values mean better predictions.

In [35]:
negative_log_likelihood = 0.0
n_tokens = len(val_data) - 1

for t1, t2 in zip(val_data[:-1], val_data[1:]):
    prob = P[t1, t2]
    negative_log_likelihood += -torch.log(prob)

avg_nll = negative_log_likelihood / n_tokens
print(f"Validation Average Negative Log-Prob: {avg_nll.item():.4f}")

Validation Average Negative Log-Prob: 2.4818


## Loss Averaging

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval() # set model to evaluation mode

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
# Complete script

# 1. Hyperparameters & Setup
batch_size = 512
block_size = 8
max_iters = 1000
eval_interval = 200
eval_iters = 100
learning_rate = 1e-2

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

# 90% train, 10% validation split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# Helper function to generate data batches
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x, y

# 2. Bigram Model Definition
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B, T, C)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            
        return logits, loss

model = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# 3. Evaluation Function
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# 4. Training Loop with Periodic Evaluation
for iter in range(max_iters):
    # Evaluate loss every few iterations
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch
    xb, yb = get_batch('train')

    # Forward pass, compute loss
    logits, loss = model(xb, yb)
    
    # Backward pass & update
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Training finished!")